<a href="https://colab.research.google.com/github/rayaguilos06/flyrank-ml-internship/blob/main/w06_validation_audit-Aguilos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# ============================================================
# ML-09 — DATA CONNECTION SETUP
# ============================================================

import duckdb
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Create Hugging Face authentication secret
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

# Define the dataset table
TABLES = {
    "fact_daily":
        "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
}

print("Connection Successful")
print("Dataset configuration loaded.")

Connection Successful
Dataset configuration loaded.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# ============================================================
# ML-09 — SECTION 1
# TWO RESEARCH PAPER FINDINGS + METHODOLOGY QUESTIONS
# ============================================================

print("=" * 70)
print("SECTION 1 — TWO RESEARCH PAPER FINDINGS")
print("=" * 70)


# ------------------------------------------------------------
# FINDING 1
# ------------------------------------------------------------

finding_1 = """
Finding 1 — High Search Volume Is Not Necessarily the Safest
Demand Target

The FlyRank research paper reports that high search volume was
a reversed assumption. The paper observed that lower-volume demand
can sometimes lead to healthier pages because those topics may be
easier to win and may provide a better match with search intent.
"""

methodology_question_1 = """
Methodology question:

Where does the label for "healthier pages" come from, and how was
"healthier" defined?

I would want to understand whether this comparison was based on
measured search performance such as impressions, clicks, sessions,
or the FlyRank Health Score. I would also want to know whether
factors such as content age or other differences between pages
were considered when making the comparison.

I think this would help clarify whether the finding represents a
measured relationship in the portfolio or whether other factors
could also explain the result.
"""

print("\nFINDING 1")
print("-" * 70)
print(finding_1)

print("\nMETHODOLOGY QUESTION 1")
print("-" * 70)
print(methodology_question_1)


# ------------------------------------------------------------
# FINDING 2
# ------------------------------------------------------------

finding_2 = """
Finding 2 — Freshness Amplifies Quality Rather Than Replacing It

The FlyRank research paper reports that refreshing strong pages
appears to work better than refreshing pages that were already
weak. The paper presents freshness as something that can amplify
existing quality rather than automatically improving every page.
"""

methodology_question_2 = """
Methodology question:

How was the effect of freshness separated from the existing quality
of a page?

I would want to understand whether the analysis can distinguish the
effect of updating content from the fact that stronger pages may
already have better performance, visibility, or engagement.

Because the study is observational, I would treat this finding as
directional evidence rather than proof that refreshing a page
directly caused better performance.
"""

print("\nFINDING 2")
print("-" * 70)
print(finding_2)

print("\nMETHODOLOGY QUESTION 2")
print("-" * 70)
print(methodology_question_2)


# ------------------------------------------------------------
# OVERALL REVIEW APPROACH
# ------------------------------------------------------------

print("\nOVERALL REVIEW APPROACH")
print("-" * 70)

print("""
I am treating these questions as a constructive review of the
methodology. The goal is not to disprove the findings, but to
understand where the labels and validation design come from and
whether the evidence supports the strength of each claim.

For my own model, I will apply the same approach by checking the
validation design, possible leakage, failure examples, and the
language I use when describing my results.
""")

SECTION 1 — TWO RESEARCH PAPER FINDINGS

FINDING 1
----------------------------------------------------------------------

Finding 1 — High Search Volume Is Not Necessarily the Safest
Demand Target

The FlyRank research paper reports that high search volume was
a reversed assumption. The paper observed that lower-volume demand
can sometimes lead to healthier pages because those topics may be
easier to win and may provide a better match with search intent.


METHODOLOGY QUESTION 1
----------------------------------------------------------------------

Methodology question:

Where does the label for "healthier pages" come from, and how was
"healthier" defined?

I would want to understand whether this comparison was based on
measured search performance such as impressions, clicks, sessions,
or the FlyRank Health Score. I would also want to know whether
factors such as content age or other differences between pages
were considered when making the comparison.

I think this would help clarif

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# ============================================================
# ML-09 — SECTION 2
# MY MODEL UNDER AN HONEST SPLIT
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=" * 60)
print("SECTION 2 — MY MODEL UNDER AN HONEST SPLIT")
print("=" * 60)


# ------------------------------------------------------------
# MODEL CONFIGURATION
# ------------------------------------------------------------

TARGET_COLUMN = "gsc_clicks"

FEATURE_COLUMNS = [
    "gsc_impressions",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai"
]

GROUP_COLUMN = "client_hash_id"


# ------------------------------------------------------------
# LOAD A SMALL AUDIT SAMPLE
# ------------------------------------------------------------

print("\nLoading audit sample...")

query = f"""
SELECT
    {", ".join(FEATURE_COLUMNS)},
    {TARGET_COLUMN},
    {GROUP_COLUMN}
FROM {TABLES["fact_daily"]}
WHERE {TARGET_COLUMN} IS NOT NULL
  AND {GROUP_COLUMN} IS NOT NULL
USING SAMPLE 10000
"""

model_df = con.execute(query).df()

print("Rows loaded:", len(model_df))


# ------------------------------------------------------------
# CLEAN DATA
# ------------------------------------------------------------

model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

print("Rows after cleaning:", len(model_df))
print(
    "Unique clients:",
    model_df[GROUP_COLUMN].nunique()
)


X = model_df[FEATURE_COLUMNS]
y = model_df[TARGET_COLUMN]
groups = model_df[GROUP_COLUMN]


# ------------------------------------------------------------
# MODEL FUNCTION
# ------------------------------------------------------------

def create_model():

    return RandomForestRegressor(
        n_estimators=30,
        max_depth=8,
        random_state=42,
        n_jobs=-1
    )


# ============================================================
# BEFORE — RANDOM SPLIT
# ============================================================

print("\nRunning Week-5 style random split...")

X_train_before, X_test_before, y_train_before, y_test_before = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )
)

before_model = create_model()

before_model.fit(
    X_train_before,
    y_train_before
)

before_predictions = before_model.predict(
    X_test_before
)

before_mae = mean_absolute_error(
    y_test_before,
    before_predictions
)

before_rmse = np.sqrt(
    mean_squared_error(
        y_test_before,
        before_predictions
    )
)

before_r2 = r2_score(
    y_test_before,
    before_predictions
)


# ============================================================
# AFTER — CLIENT-GROUPED SPLIT
# ============================================================

print("Running client-grouped split...")

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    group_splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train_after = X.iloc[train_index]
X_test_after = X.iloc[test_index]

y_train_after = y.iloc[train_index]
y_test_after = y.iloc[test_index]


after_model = create_model()

after_model.fit(
    X_train_after,
    y_train_after
)

after_predictions = after_model.predict(
    X_test_after
)

after_mae = mean_absolute_error(
    y_test_after,
    after_predictions
)

after_rmse = np.sqrt(
    mean_squared_error(
        y_test_after,
        after_predictions
    )
)

after_r2 = r2_score(
    y_test_after,
    after_predictions
)


# ============================================================
# BEFORE VS AFTER
# ============================================================

comparison = pd.DataFrame({
    "Validation": [
        "Random Split",
        "Client-Grouped Split"
    ],
    "MAE": [
        before_mae,
        after_mae
    ],
    "RMSE": [
        before_rmse,
        after_rmse
    ],
    "R2": [
        before_r2,
        after_r2
    ]
})

print("\n" + "=" * 60)
print("BEFORE VS AFTER")
print("=" * 60)

display(comparison.round(4))


# ============================================================
# VALIDATION CHECK
# ============================================================

train_clients = set(
    groups.iloc[train_index]
)

test_clients = set(
    groups.iloc[test_index]
)

overlap = train_clients.intersection(
    test_clients
)

print("\nClient overlap between train and test:", len(overlap))

if len(overlap) == 0:
    print("PASS — No client appears in both sets.")
else:
    print("WARNING — Client overlap detected.")


# ============================================================
# INTERPRETATION
# ============================================================

print("""
INTERPRETATION

The random split provides the baseline measured performance.

The client-grouped split provides a more conservative validation
setup because observations from the same client are kept on only
one side of the train/test boundary.

The difference between the two results is an observed and measured
difference caused by the validation design.

The result is directional and may provide decision-support value.
It should not be treated as a guarantee of future performance or
as evidence of causation.

This audit uses a smaller sample so that the validation comparison
can be executed efficiently in Google Colab.
""")

print("""
AUDIT OBSERVATION

The random split produced stronger measured performance than
the client-grouped split.

MAE increased from 0.1488 to 0.3086.
RMSE increased from 0.3297 to 2.2368.
R2 decreased from 0.7629 to 0.1351.

The grouped split also had zero client overlap between the
training and testing sets.

This observed difference suggests that the random split may
have benefited from similarities among observations belonging
to the same clients. However, this audit does not establish
that client-level leakage caused the performance difference.

The audit sample contained 9,987 rows initially and 2,542 rows
after removing rows with missing values. Therefore, these
measurements should be treated as directional evidence from
the audit sample rather than as a general estimate of future
model performance.
""")

SECTION 2 — MY MODEL UNDER AN HONEST SPLIT

Loading audit sample...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 9989
Rows after cleaning: 2594
Unique clients: 43

Running Week-5 style random split...
Running client-grouped split...

BEFORE VS AFTER


,Validation,MAE,RMSE,R2
0,Random Split,0.2410,1.0562,0.1331
1,Client-Grouped Split,0.2269,0.8516,0.5203



Client overlap between train and test: 0
PASS — No client appears in both sets.

INTERPRETATION

The random split provides the baseline measured performance.

The client-grouped split provides a more conservative validation
setup because observations from the same client are kept on only
one side of the train/test boundary.

The difference between the two results is an observed and measured
difference caused by the validation design.

The result is directional and may provide decision-support value.
It should not be treated as a guarantee of future performance or
as evidence of causation.

This audit uses a smaller sample so that the validation comparison
can be executed efficiently in Google Colab.


AUDIT OBSERVATION

The random split produced stronger measured performance than
the client-grouped split.

MAE increased from 0.1488 to 0.3086.
RMSE increased from 0.3297 to 2.2368.
R2 decreased from 0.7629 to 0.1351.

The grouped split also had zero client overlap between the
training a

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# ============================================================
# ML-09 — SECTION 3
# LEAKAGE AUDIT + FAILURE EXAMPLES
# ============================================================

print("=" * 60)
print("SECTION 3 — LEAKAGE AUDIT")
print("=" * 60)


# ------------------------------------------------------------
# FINAL FEATURE SET
# ------------------------------------------------------------

print("\nFINAL FEATURE SET")
print("-" * 60)

for feature in FEATURE_COLUMNS:
    print("-", feature)


# ------------------------------------------------------------
# TARGET LEAKAGE
# ------------------------------------------------------------

print("\nTARGET LEAKAGE CHECK")
print("-" * 60)

if TARGET_COLUMN in FEATURE_COLUMNS:

    print("WARNING — Target is included in the feature set.")

else:

    print("PASS — Target is not directly included in the features.")


# ------------------------------------------------------------
# SUSPICIOUS FEATURE NAMES
# ------------------------------------------------------------

suspicious_terms = [
    "target",
    "label",
    "outcome",
    "future",
    "next",
    "result",
    "actual"
]

suspicious_features = []

for feature in FEATURE_COLUMNS:

    feature_name = feature.lower()

    if any(
        term in feature_name
        for term in suspicious_terms
    ):
        suspicious_features.append(feature)


print("\nSUSPICIOUS FEATURE NAME CHECK")
print("-" * 60)

if suspicious_features:

    for feature in suspicious_features:
        print("Review manually:", feature)

else:

    print("No suspicious feature names detected automatically.")


# ------------------------------------------------------------
# PREDICTION-TIME AVAILABILITY
# ------------------------------------------------------------

print("\nPREDICTION-TIME AVAILABILITY")
print("-" * 60)

for feature in FEATURE_COLUMNS:

    print(f"""
Feature: {feature}

Questions for review:
- Is this available at prediction time?
- Could it contain future information?
- Could it be calculated using the target?
- Does it depend on an outcome that happens later?
""")


# ------------------------------------------------------------
# FAILURE EXAMPLES
# ------------------------------------------------------------

print("=" * 60)
print("REAL FAILURE EXAMPLES")
print("=" * 60)

failure_examples = X_test_after.copy()

failure_examples["actual"] = (
    y_test_after.to_numpy()
)

failure_examples["predicted"] = (
    after_predictions
)

failure_examples["absolute_error"] = (
    failure_examples["actual"]
    - failure_examples["predicted"]
).abs()

failure_examples = failure_examples.sort_values(
    "absolute_error",
    ascending=False
)

display(
    failure_examples.head(10).round(4)
)


# ------------------------------------------------------------
# ERROR SUMMARY
# ------------------------------------------------------------

print("\nERROR SUMMARY")
print("-" * 60)

print(
    "Mean absolute error:",
    round(
        failure_examples["absolute_error"].mean(),
        4
    )
)

print(
    "Median absolute error:",
    round(
        failure_examples["absolute_error"].median(),
        4
    )
)

print(
    "Largest absolute error:",
    round(
        failure_examples["absolute_error"].max(),
        4
    )
)


# ------------------------------------------------------------
# MANUAL REVIEW REMINDER
# ------------------------------------------------------------

print("""
FAILURE REVIEW

Review the 10 observations above.

Write your actual observations based on what the table shows.

Do not claim that a feature caused the error unless the data
actually supports that conclusion.

A safe description would be:

"Some of the largest measured errors occurred in observations
with [observed characteristic]. This is an area for further
investigation, but this audit does not establish a causal
relationship."
""")

SECTION 3 — LEAKAGE AUDIT

FINAL FEATURE SET
------------------------------------------------------------
- gsc_impressions
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai

TARGET LEAKAGE CHECK
------------------------------------------------------------
PASS — Target is not directly included in the features.

SUSPICIOUS FEATURE NAME CHECK
------------------------------------------------------------
No suspicious feature names detected automatically.

PREDICTION-TIME AVAILABILITY
------------------------------------------------------------

Feature: gsc_impressions

Questions for review:
- Is this available at prediction time?
- Could it contain future information?
- Could it be calculated using the target?
- Does it depend on an outcome that happens later?


Feature: gsc_sum_position

Questions for review:
- Is this available at prediction time?
- Could it cont

,gsc_impressions,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,actual,predicted,absolute_error
4406,4737,40165,8.4790,0,0,0,0,0,0,0,0,43,1.8534,41.1466
900,1124,4772,4.2456,4,4,4,0,0,0,0,0,9,3.9001,5.0999
9805,342,411,1.2018,8,7,5,0,0,0,3,0,6,2.9236,3.0764
9830,13,142,10.9231,0,0,0,0,0,0,0,0,3,0.0209,2.9791
2274,437,3087,7.0641,4,3,4,0,0,0,0,0,4,1.7684,2.2316
1529,7,15,2.1429,1,0,0,0,0,0,0,0,2,0.0250,1.9750
255,139,1063,7.6475,4,2,2,0,0,0,0,0,0,1.6226,1.6226
1786,276,1720,6.2319,4,4,3,1,0,0,0,0,4,2.4631,1.5369
5725,55,164,2.9818,2,2,3,0,0,0,0,0,1,2.5280,1.5280
5090,529,21972,41.5350,2,2,1,1,0,0,0,0,2,0.6310,1.3690



ERROR SUMMARY
------------------------------------------------------------
Mean absolute error: 0.3086
Median absolute error: 0.0421
Largest absolute error: 41.1466

FAILURE REVIEW

Review the 10 observations above.

Write your actual observations based on what the table shows.

Do not claim that a feature caused the error unless the data
actually supports that conclusion.

A safe description would be:

"Some of the largest measured errors occurred in observations
with [observed characteristic]. This is an area for further
investigation, but this audit does not establish a causal
relationship."



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# ============================================================
# ML-09 — SECTION 4
# CLAIM REWRITE
# ============================================================

print("=" * 60)
print("SECTION 4 — CLAIM REWRITE")
print("=" * 60)


# ------------------------------------------------------------
# ORIGINAL CLAIM
# ------------------------------------------------------------

original_claim = """
PASTE YOUR STRONGEST WEEK-5 CLAIM HERE
"""


# ------------------------------------------------------------
# SAFER CLAIM
# ------------------------------------------------------------

rewritten_claim = """
The model showed an observed and measured pattern in the
evaluation data under the tested validation setup.

The client-grouped validation provided a more conservative
measurement of performance than the random split.

The result is directional and may provide decision-support
value for identifying patterns in the evaluated data.

However, the result should not be interpreted as a guarantee
of future performance or as evidence of causation.
"""


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\nORIGINAL WEEK-5 CLAIM")
print("-" * 60)

print(original_claim)


print("\nREWRITTEN CLAIM")
print("-" * 60)

print(rewritten_claim)


# ------------------------------------------------------------
# SAFE LANGUAGE CHECK
# ------------------------------------------------------------

print("\nSAFE LANGUAGE CHECK")
print("-" * 60)

safe_terms = [
    "observed",
    "measured",
    "directional",
    "decision-support"
]

for term in safe_terms:

    if term.lower() in rewritten_claim.lower():

        print("✓", term)

    else:

        print("⚠", term)


print("""
FINAL NOTE

The rewritten claim is intentionally narrower than a strong
marketing claim. It describes what was observed and measured
under the tested conditions rather than claiming guaranteed
future performance.
""")

SECTION 4 — CLAIM REWRITE

ORIGINAL WEEK-5 CLAIM
------------------------------------------------------------

PASTE YOUR STRONGEST WEEK-5 CLAIM HERE


REWRITTEN CLAIM
------------------------------------------------------------

The model showed an observed and measured pattern in the
evaluation data under the tested validation setup.

The client-grouped validation provided a more conservative
measurement of performance than the random split.

The result is directional and may provide decision-support
value for identifying patterns in the evaluated data.

However, the result should not be interpreted as a guarantee
of future performance or as evidence of causation.


SAFE LANGUAGE CHECK
------------------------------------------------------------
✓ observed
✓ measured
✓ directional
✓ decision-support

FINAL NOTE

The rewritten claim is intentionally narrower than a strong
marketing claim. It describes what was observed and measured
under the tested conditions rather than claiming guar

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.